## Importando as libs

In [ ]:
import os
from pathlib import Path

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.prompts import ChatPromptTemplate

import time
import re

EMBEDDINGS_PDFS_PATH = "/home/kiev/Documents/DECOMPilot/EmbeddingsSource"
API_KEY = ""
VECTORSTORE_PATH = "./chroma_db"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
K_ = 5
QUERY = "How many steps in decomission process and what are they?"

/home/kiev/Dev/decompilot/pdf-similarity-search/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Fazendo os embeddings

In [3]:
def load_pdf(pdf_path: str):
    print(f"PDF: {pdf_path}")

    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    print(f"\tCarregado ({len(documents)} páginas)")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        length_function=len,
    )
    
    chunks = text_splitter.split_documents(documents)
    print(f"\tCHUNKS: {len(chunks)}\n")
    
    for chunk in chunks:
        chunk.metadata['source'] = pdf_path
        chunk.metadata['filename'] = os.path.basename(pdf_path)
    
    return chunks

In [4]:
directory = Path(EMBEDDINGS_PDFS_PATH)
pdf_files = list(directory.glob("*.pdf"))

if not pdf_files:
    print(f"Nenhum arquivo PDF encontrado")

print(f"{len(pdf_files)} PDFs encontrados")

all_documents = []

for pdf_file in pdf_files:
    documents = load_pdf(str(pdf_file))
    all_documents.extend(documents)

print("PDFs carregados.")

23 PDFs encontrados
PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/2018_Decom_Guidance_Notes_November.pdf
	Carregado (138 páginas)
	CHUNKS: 405

PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/2019_OGUK-Decommissioning-Work-Breakdown-Structure-Guidelines.pdf
	Carregado (13 páginas)
	CHUNKS: 36

PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/Resolucao-852 _23-de-setembro-ANP.pdf
	Carregado (19 páginas)
	CHUNKS: 298

PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/2019_Clarissa_Marcelo_Renato-Decommissioning in Brazil_ legal aspects of a technical analysis.pdf
	Carregado (10 páginas)
	CHUNKS: 52

PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/2023_Plug and abandonment of oil and gas wells – A comprehensive review of regulations, practices, and related impact of materials selection .pdf
	Carregado (28 páginas)
	CHUNKS: 256

PDF: /home/kiev/Documents/DECOMPilot/EmbeddingsSource/2022_ IBP_ Transferencia-de-ativos-integridade-de-pocos.pdf
	Carregado (18 págin

In [5]:
print(f"\nCriando Vectorstore com {len(all_documents)} documentos")

embeddings = GoogleGenerativeAIEmbeddings(
    google_api_key=API_KEY,
    model="models/embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    persist_directory=VECTORSTORE_PATH
)

print(f"Vectorstore criado")


Criando Vectorstore com 4551 documentos
Vectorstore criado


## Fazendo pergunta à LLM com contexto por similarity search

In [12]:
llm = ChatGoogleGenerativeAI(
    google_api_key=API_KEY,
    model="models/gemini-2.0-flash",
    temperature=0.3
)

In [14]:
def get_answer(query: str = QUERY, k: int = K_) -> str:
    relevant_docs = vectorstore.similarity_search(query, k=k)
    print(f"Encontrados {len(relevant_docs)} documentos relevantes")

    context_text = "\n\n".join([
        f"Fonte: {doc.metadata.get('filename', 'Unknown')} - Página {doc.metadata.get('page', 'N/A')}\nConteúdo: {doc.page_content}"
        for doc in relevant_docs
    ])

    system_prompt = """
    Você é um assistente especializado em analisar documentos PDF. 
    Sua tarefa é responder perguntas dos usuários baseando-se APENAS no contexto fornecido.

    Regras importantes:
    1. Responda APENAS com base nas informações fornecidas no contexto
    2. Se a informação não estiver no contexto, diga claramente que não tem essa informação
    3. Seja preciso e direto nas respostas
    4. Cite as fontes quando relevante
    5. Responda em português brasileiro
    6. Se houver informações conflitantes, mencione isso
    7. Organize a resposta de forma clara e estruturada

    Contexto dos documentos:
    {context}

    Pergunta do usuário: {question}

    Responda de forma clara, útil e bem estruturada:
    """
            
    prompt = ChatPromptTemplate.from_template(system_prompt)

    messages = prompt.format_messages(
        context=context_text,
        question=query
    )

    response = llm.invoke(messages)

    return response, relevant_docs

In [15]:
answer, relevant_docs = get_answer()

print(f"Pergunta: {QUERY}\n")
print(f"Resposta:")
print(f"{answer.content}\n")

if relevant_docs:
    print(f"Fontes ({len(relevant_docs)} documentos):")
    for i, source in enumerate(relevant_docs, 1):
        print(f"\n\t{i}. {source.metadata.get('filename', 'Unknown')} - Página {source.metadata.get('page', 'N/A')}")
        print(f"\t\t{source.page_content[:200]}...\n")

Encontrados 5 documentos relevantes
Pergunta: How many steps in decomission process and what are they?

Resposta:
De acordo com o documento "2018_Decom_Guidance_Notes_November.pdf", existem cinco estágios principais no processo de descomissionamento, começando antes da cessação da produção e continuando através da identificação inicial de opções, até a avaliação detalhada e redação de um programa de descomissionamento (DP), seguido pela execução (página 21).

O Anexo H também detalha os estágios do processo de descomissionamento, incluindo discussões detalhadas entre o operador e o OPRED, submissão de um rascunho de consulta do programa, consultas estatutárias pelo operador, aprovação do programa pelo Secretário de Estado e execução do descomissionamento pelo operador (página 132).

Fontes (5 documentos):

	1. 2018_Decom_Guidance_Notes_November.pdf - Página 21
		5. Planning for Decommissioning 
21 
5. Planning for Decommissioning 
The Decommissioning Programme Process 
 This section ou

## Evaluation

In [16]:
evaluation_script = [
    {
        "id": 1,
        "question": "What are the three different statuses of a well once downhole activities or production are discontinued?",
        "ideal_answer": (
            "Os três status são:\n"
            "- **Suspensão:** O equipamento de controle do poço não é removido.\n"
            "- **Abandono Temporário:** O equipamento de controle do poço é removido com a intenção de reentrada posterior ou abandono permanente.\n"
            "- **Abandono Permanente:** O poço, ou parte dele, é tamponado e abandonado com a intenção de nunca mais ser reutilizado ou reentrado."
        ),
        "keywords": ["suspensão", "abandono temporário", "abandono permanente", "equipamento", "tamponado"]
    },
    {
        "id": 2,
        "question": "What is the primary objective of a plug and abandonment (P&A) operation and why is it necessary?",
        "ideal_answer": (
            "O principal objetivo de uma operação de P&A é restaurar a funcionalidade da rocha de capeamento para garantir a integridade do poço permanentemente. "
            "A operação é necessária para estabelecer barreiras que impeçam o fluxo de fluidos perigosos para os arredores, como o ambiente marinho, águas subterrâneas, o solo ou a atmosfera."
        ),
        "keywords": ["objetivo", "integridade", "barreiras", "fluidos perigosos", "ambiente"],
    },
    {
        "id": 3,
        "question": "Explain the 'two-barrier' philosophy (or 'hat-over-hat' principle) in the context of well P&A.",
        "ideal_answer": (
            "A filosofia de duas barreiras estipula que o poço deve ser equipado com duas barreiras de poço independentes: uma barreira primária e uma secundária. "
            "A barreira primária é o primeiro invólucro que impede o fluxo de uma fonte potencial. "
            "A barreira secundária atua como um backup para a barreira primária caso ela falhe. "
            "Este conceito também é conhecido como o princípio 'chapéu sobre chapéu'."
        ),
        "keywords": ["duas barreiras", "primária", "secundária", "backup", "chapéu sobre chapéu"],
    },
    {
        "id": 4,
        "question": "List five common challenges associated with P&A operations mentioned in the document.",
        "ideal_answer": (
            "Cinco desafios comuns mencionados são: altas temperaturas, formações não consolidadas, alterações na resistência da formação devido à depleção, "
            "estresse tectônico (como cisalhamento e subsidência), pressão sustentada no revestimento (SCP), falta de dados de poços antigos e a dificuldade "
            "de verificar o cimento atrás da segunda coluna de revestimento."
        ),
        "keywords": ["altas temperaturas", "formações não consolidadas", "depleção", "estresse tectônico", "SCP"],
    },
    {
        "id": 5,
        "question": "Compare Portland cement and thermosetting polymers as permanent plugging materials, mentioning two advantages and two limitations of each, based on the text.",
        "ideal_answer": (
            "**Cimento Portland:**\n"
            "- **Vantagens:** É o principal material de barreira usado na indústria do petróleo, sendo, portanto, bem conhecido e estudado. "
            "Existem várias classes API disponíveis para se adequar a diferentes condições de poço.\n"
            "- **Limitações:** Pode sofrer retração, o que pode criar microanéis. Pode ser frágil e degradar a longo prazo devido à exposição a altas temperaturas e substâncias químicas como H₂S e CO₂.\n\n"
            "**Polímeros Termofixos (Resinas):**\n"
            "- **Vantagens:** Possuem permeabilidade muito baixa (estanques a gás), forte adesão à formação e ao aço, e boas propriedades mecânicas.\n"
            "- **Limitações:** Geralmente são frágeis no estado sólido. A durabilidade a longo prazo é parcialmente desconhecida e pode haver retração química."
        ),
        "keywords": ["cimento portland", "polímeros termofixos", "vantagens", "limitações", "permeabilidade", "adesão"],
    }
]

In [17]:
def clean_text(text: str) -> str:
        text = re.sub(r'\[cite:\s*\d+\]', '', text)
        text = re.sub(r'\[cite_start\]', '', text)
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s\-\.\,\;\:\!\?]', '', text)
        return text.strip().lower()

In [21]:
def get_evaluation(question, ideal_answer, agent_answer):
    evaluation_prompt = """
    Você é um especialista em avaliação de qualidade de respostas. Sua tarefa é comparar uma resposta do agente com uma resposta ideal e dar um score baseado na similaridade semântica e factual.

    CRITÉRIOS DE AVALIAÇÃO:
    - **Score 0 (Totalmente Diferente)**: A resposta do agente não aborda o tema da pergunta ou fornece informações completamente incorretas/irrelevantes
    - **Score 1 (Pouco Diferente)**: A resposta aborda o tema mas com informações limitadas, imprecisas ou muito superficiais
    - **Score 2 (Similar)**: A resposta aborda bem o tema com informações corretas, mas pode estar incompleta ou ter pequenas diferenças
    - **Score 3 (Igual)**: A resposta é muito similar à ideal, cobrindo os mesmos pontos principais com precisão

    CONSIDERE:
    - Precisão factual das informações
    - Cobertura dos tópicos principais
    - Clareza e estrutura da resposta
    - Uso correto de terminologia técnica
    - Completude da resposta

    PERGUNTA: {question}

    RESPOSTA IDEAL: {ideal_answer}

    RESPOSTA DO AGENTE: {agent_answer}

    INSTRUÇÕES:
    1. Analise cuidadosamente ambas as respostas
    2. Compare o conteúdo semântico e factual
    3. Atribua um score de 0 a 3
    4. Forneça uma justificativa breve para o score

    FORMATO DA RESPOSTA:
    Score: [0-3]
    Justificativa: [explicação do score]

    Responda apenas com o score e justificativa:
    """
    
    evaluation_llm = ChatGoogleGenerativeAI(
        google_api_key=API_KEY,
        model="models/gemini-2.0-flash",
        temperature=0.1 
    )
    
    evaluation_template = ChatPromptTemplate.from_template(evaluation_prompt)
    
    messages = evaluation_template.format_messages(
        question=question,
        ideal_answer=ideal_answer,
        agent_answer=agent_answer
    )
        
    response = evaluation_llm.invoke(messages)
    evaluation_text = response.content.strip()

    return evaluation_text

In [20]:
for evaluation in evaluation_script:
    print(f"\nPergunta {evaluation['id']}: {evaluation['question']}")

    start_time = time.time()
    result, relevant_docs = get_answer(evaluation['question'], k=10)
    end_time = time.time()
    
    agent_answer = result.content
    response_time = end_time - start_time
    
    print(f"\tTempo de resposta: {response_time:.2f}s")
    print(f"\tDocumentos encontrados: {len(relevant_docs)}")

    texto1 = clean_text(agent_answer)
    texto2 = clean_text(evaluation['ideal_answer'])


    print(f"\tTexto 1: {texto1}")
    print(f"\tTexto 2: {texto2}")

    found_keywords = []
        
    for keyword in evaluation['keywords']:
        if keyword.lower() in agent_answer:
            found_keywords.append(keyword)
    
    print(f"\tSimilaridade: {get_evaluation(evaluation['question'], evaluation['ideal_answer'], agent_answer)}")
    print(f"\tPalavras-chave encontradas: {len(found_keywords)}/{len(evaluation['keywords'])}")


Pergunta 1: What are the three different statuses of a well once downhole activities or production are discontinued?
Encontrados 10 documentos relevantes
	Tempo de resposta: 4.98s
	Documentos encontrados: 10
	Texto 1: após a descontinuação das atividades de fundo de poço ou da produção, o status do poço pode ser classificado em três tipos diferentes de acordo com 2:  suspensão: o poço é submetido a construção ou intervenção, e a operação pode ser suspensa sem a remoção do equipamento de controle do poço p. 18.  temporariamente abandonado: o poço foi abandonado e o equipamento de controle do poço é removido com a intenção de reentrada posterior ou abandono permanente p. 18. outro termo para essa situação pode ser suspensão de longo prazo p. 18.  permanentemente abandonado: o poço é permanentemente tamponado e abandonado quando atinge o fim de seu ciclo de vida p. 19.
	Texto 2: os três status são: - suspensão: o equipamento de controle do poço não é removido. - abandono temporário: o eq